In [1]:
import pandas as pd
import sqlite3

conn = sqlite3.connect("../database/yieldwatch.db")
print("Connected.")

Connected.


In [2]:
with open("../sql/01_overview_metrics.sql", "r") as f:
    query = f.read()

result = pd.read_sql(query, conn)
result

,total_production,total_good_units,total_defective_units,overall_yield_pct,overall_defect_pct
0,21002956,19928675,1074281,94.89,5.11


In [5]:
print(repr(full_sql))

''


In [6]:
with open("../sql/02_breakdown_by_dimension.sql", "r") as f:
    full_sql = f.read()

print(repr(full_sql[:100]))  # just show the first 100 characters to confirm it's not empty

'-- Defect rate and yield by production line\nSELECT\n    line_id,\n    SUM(produced_units) AS total_pro'


In [7]:
queries = [q.strip() for q in full_sql.split(";") if q.strip()]
print("Number of queries found:", len(queries))

by_line = pd.read_sql(queries[0], conn)
by_machine = pd.read_sql(queries[1], conn)
by_shift = pd.read_sql(queries[2], conn)
by_material = pd.read_sql(queries[3], conn)

print("=== By Line ===")
print(by_line)
print("\n=== By Shift ===")
print(by_shift)
print("\n=== By Material ===")
print(by_material)

Number of queries found: 4
=== By Line ===
  line_id  total_production  total_defects  yield_pct  defect_pct
0     L03           3179996         185210      94.18        5.82
1     L04           1074154          58121      94.59        5.41
2     L01           7306723         384317      94.74        5.26
3     L02           5271559         261305      95.04        4.96
4     L05           4170524         185328      95.56        4.44

=== By Shift ===
  shift_id  total_production  total_defects  defect_pct
0    Night           7047303         422443        5.99
1  Morning           6978801         325910        4.67
2  Evening           6976852         325928        4.67

=== By Material ===
  material_id  total_production  total_defects  defect_pct
0       MAT09           2073729         131285        6.33
1       MAT03           2113091         133278        6.31
2       MAT06           2100602         102453        4.88
3       MAT04           2113196         102717        4.86
4  

In [8]:
by_machine

,machine_id,total_production,total_defects,defect_pct
0,M07,1052772,83398,7.92
1,M14,1051137,76013,7.23
2,M18,1065307,65793,6.18
3,M17,1051801,60973,5.80
4,M12,1033658,59679,5.77
5,M02,1060276,61029,5.76
6,M11,1056441,60297,5.71
7,M09,1074154,58121,5.41
8,M13,1065865,57311,5.38
9,M01,1045141,53708,5.14


In [10]:
with open("../sql/03_defect_types_and_trends.sql", "r") as f:
    full_sql = f.read()

queries = [q.strip() for q in full_sql.split(";") if q.strip()]
print("Number of queries found:", len(queries))

defect_types = pd.read_sql(queries[0], conn)
daily_trend = pd.read_sql(queries[1], conn)

print("=== Defect Types ===")
print(defect_types)
print("\n=== Daily Trend (first 10 days) ===")
print(daily_trend.head(10))
print("\nDaily trend row count:", len(daily_trend))

Number of queries found: 2
=== Defect Types ===
              defect_type  defect_count  pct_of_all_defects
0          Surface Damage         10127               16.89
1             Overheating         10078               16.81
2         Alignment Error         10036               16.74
3         Dimension Error          9976               16.64
4           Surface Crack          9886               16.49
5  Material Contamination          9856               16.44

=== Daily Trend (first 10 days) ===
  production_date  daily_production  daily_defects  daily_defect_pct
0      2025-09-01             57133           2629              4.60
1      2025-09-02             64434           2879              4.47
2      2025-09-03             56416           2892              5.13
3      2025-09-04             61309           2941              4.80
4      2025-09-05             61000           2722              4.46
5      2025-09-06             56443           2540              4.50
6      2025-

In [11]:
with open("../sql/03_defect_types_and_trends.sql", "r") as f:
    full_sql = f.read()

queries = [q.strip() for q in full_sql.split(";") if q.strip()]
print("Number of queries found:", len(queries))

weekly = pd.read_sql(queries[2], conn)
rolling = pd.read_sql(queries[3], conn)

print("=== Weekly Defect Rate (first 8 weeks) ===")
print(weekly.head(8))
print("\n=== Rolling Windows (rows 25-35, so 30-day window is populated) ===")
print(rolling.iloc[25:35])

Number of queries found: 4
=== Weekly Defect Rate (first 8 weeks) ===
  year_week  weekly_production  weekly_defects  weekly_defect_pct
0   2025-35             409252           19068               4.66
1   2025-36             383592           17963               4.68
2   2025-37             401148           18538               4.62
3   2025-38             401030           18840               4.70
4   2025-39             384559           17781               4.62
5   2025-40             398678           18717               4.69
6   2025-41             381909           18369               4.81
7   2025-42             430092           19835               4.61

=== Rolling Windows (rows 25-35, so 30-day window is populated) ===
   production_date  daily_production  daily_defects  rolling_7day_defect_pct  \
25      2025-09-26             54451           2775                     4.76   
26      2025-09-27             59148           2749                     4.74   
27      2025-09-28         

In [12]:
with open("../sql/04_risk_combinations.sql", "r") as f:
    full_sql = f.read()

queries = [q.strip() for q in full_sql.split(";") if q.strip()]
print("Number of queries found:", len(queries))

machine_shift = pd.read_sql(queries[0], conn)
machine_material = pd.read_sql(queries[1], conn)
triple_combo = pd.read_sql(queries[2], conn)
risk_ranked = pd.read_sql(queries[3], conn)

print("=== Top Machine+Shift Risk Combos ===")
print(machine_shift.head(10))
print("\n=== Top Risk-Ranked Combos (vs baseline) ===")
print(risk_ranked)


Number of queries found: 4
=== Top Machine+Shift Risk Combos ===
  machine_id shift_id  total_production  total_defects  defect_pct
0        M07    Night            354222          35612       10.05
1        M14    Night            348143          32311        9.28
2        M18    Night            364596          29621        8.12
3        M07  Morning            349322          24290        6.95
4        M07  Evening            349228          23496        6.73
5        M11    Night            360280          23402        6.50
6        M12    Night            350790          22782        6.49
7        M17    Night            349137          22584        6.47
8        M02    Night            352162          22470        6.38
9        M14  Morning            341467          21508        6.30

=== Top Risk-Ranked Combos (vs baseline) ===
   machine_id shift_id material_id  total_production  defect_pct  \
0         M07    Night       MAT09             34231       11.27   
1         M07   